# 06. App Predictions

CRISP-DM phase: Deployment

This notebook prepares the two files that the Streamlit dashboard reads, so the app does not need to load the model:

- `app/data/app_predictions.csv`: for every county and every month from January 2020 to December 2024, the flood risk for the **next** month, the alert level (Warning, Watch or No alert), and whether a flood was recorded. It also has the risk and alert level that the **live** forecast would have given (without the flood records of the data month), in the columns `risk_live` and `level_live`.
- `app/data/kenya_counties_simple.geojson`: the 47 county shapes, simplified so the file is small and the map is fast.

The final model is the XGBoost model of `XGBoost.ipynb`. Nothing is trained or changed here.

## 1. Setup

### 1.1 Importing Libraries

In [1]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import MultiPolygon
from shapely.geometry.polygon import orient

### 1.2 Loading the Final Model and Its Settings

The model and its two thresholds were saved by `XGBoost.ipynb`. A month gets a **Warning** when the risk is at least the warning threshold (the highest threshold with 70% out-of-fold recall) and a **Watch** when the risk is at least the watch threshold (80% out-of-fold recall) but below the warning threshold. Every Warning county is also above the Watch line, because the warning threshold is the higher one.

In [2]:
#loading the final xgboost model and its settings
model = joblib.load('../models/xgboost.joblib')
with open('../models/xgboost_settings.json') as settings_file:
    settings = json.load(settings_file)

In [3]:
#the two thresholds
watch_threshold = settings['watch_threshold']
warning_threshold = settings['warning_threshold']
print('watch', round(watch_threshold, 4))
print('warning', round(warning_threshold, 4))

watch 0.3425
warning 0.4374


In [4]:
#the 25 features of the model, in the order the model was trained with
features = list(model.feature_names_in_)
print(len(features))
print(features)

25
[np.str_('flood_this_month'), np.str_('rain_mm'), np.str_('rain_lag1'), np.str_('rain_lag2'), np.str_('rain_pct_normal'), np.str_('neighbour_rain'), np.str_('ndvi_pct_normal'), np.str_('oni'), np.str_('dmi'), np.str_('month_sin'), np.str_('month_cos'), np.str_('soil_moisture_l1'), np.str_('runoff_rank'), np.str_('discharge_rank'), np.str_('pct_flat'), np.str_('area_km2'), np.str_('lc_tree'), np.str_('lc_grass'), np.str_('lc_crop'), np.str_('lc_bare'), np.str_('pct_within_1km_river'), np.str_('pct_water'), np.str_('pct_occasional_water'), np.str_('pop_density'), np.str_('county_flood_rate')]


## 2. Loading the Full Table and Keeping January 2020 to December 2024

The app shows the months from **January 2020 to December 2024**. The full table also has the months of 2025, but they are left out for two reasons:

- the model uses `flood_this_month` as a feature, and the flood records of 2025 are almost empty (only one flood is recorded in the whole year), so the model would see "no flood this month" almost everywhere and the 2025 predictions would be too low;
- the labels of 2025 are incomplete, so the predictions could not be compared with what happened.

The rows of December 2024 forecast January 2025, whose flood records may also be incomplete. The app marks them with `records_complete = False`.

In [5]:
#loading the full table (pcode as text)
table = pd.read_csv('../data/model_table_full.csv', dtype={'pcode': str})
print(table.shape)

(14664, 70)


In [6]:
#keeping the rows from january 2020 to december 2024
table = table[(table['year'] >= 2020) & (table['year'] <= 2024)].copy()
table = table.sort_values(['year', 'month', 'pcode']).reset_index(drop=True)
print(table.shape)
print(table['pcode'].nunique(), 'counties')
print(table.groupby(['year', 'month']).ngroups, 'months')

(2820, 70)
47 counties
60 months


#### Observation

The full table has 14,664 rows (47 counties x 312 months). After keeping January 2020 to December 2024 there are 2,820 rows: 47 counties x 60 months. These are the same rows as the test set of the model notebooks.

## 3. Replacing `county_flood_rate`

The `county_flood_rate` column of the full table uses the flood labels up to 2019 on every row, and it was calculated before the training-only fix of `01_preprocessing.ipynb`. The model was trained with the flood rate of each county from the **training rows only**, which is saved in `data/train.csv`. So here the column is replaced with the training-only rate of each county (it has the same value on every row of a county), mapped by `pcode`.

In [7]:
#the training-only flood rate of each county, from the training set the model was trained with
train = pd.read_csv('../data/train.csv', dtype={'pcode': str})
print(train.groupby('pcode')['county_flood_rate'].nunique().max(), 'different value(s) per county in train')
rate_by_county = train.groupby('pcode')['county_flood_rate'].first()
print(len(rate_by_county), 'counties')

1 different value(s) per county in train


47 counties


In [8]:
#replacing county_flood_rate with the training-only rate, mapped by pcode
old_rate = table['county_flood_rate'].copy()
table['county_flood_rate'] = table['pcode'].map(rate_by_county)
print('missing after the mapping', table['county_flood_rate'].isna().sum())
print('rows where the value changed', (old_rate.round(6) != table['county_flood_rate'].round(6)).sum())
print('largest change', round((old_rate - table['county_flood_rate']).abs().max(), 4))

missing after the mapping 0
rows where the value changed 2820
largest change 0.0308


#### Observation

The training set has one `county_flood_rate` value for each county (47 counties), and the mapping by `pcode` leaves no missing value. All 2,820 rows changed: the training-only rate is different from the rate in the full table, by up to 0.031.

## 4. Calculating `month_sin` and `month_cos` Again

The csv file of the full table stores numbers with fewer digits than the training files. This matters for `month_cos` in March and September, where the true value is almost 0: the file has `-1.84e-16` for September, but the model was trained with `-1.8369701987210297e-16`, and one of the splits of the trees sits exactly at that value. With the rounded value, all the September rows go to the other side of the split and their risk changes by up to 0.25. So `month_sin` and `month_cos` are calculated again from the month, with the same formula as the data preparation, which gives the exact training values.

In [9]:
#month_cos of september in the csv file, and calculated again from the month
print('in the table:', table.loc[table['month'] == 9, 'month_cos'].iloc[0])
table['month_sin'] = np.sin(2 * np.pi * table['month'] / 12)
table['month_cos'] = np.cos(2 * np.pi * table['month'] / 12)
print('calculated:', table.loc[table['month'] == 9, 'month_cos'].iloc[0])

in the table: -1.84e-16
calculated: -1.8369701987210297e-16


## 5. Checking the 25 Features

In [10]:
#the feature table, in the exact order of the model
X = table[features]
print(X.shape)
print(list(X.columns) == list(model.feature_names_in_))

(2820, 25)
True


In [11]:
#missing values in the features
print('missing values in the features:', X.isna().sum().sum())

missing values in the features: 0


In [12]:
#the same rows in test.csv (the test set of the model notebooks) must have the same feature values
test = pd.read_csv('../data/test.csv', dtype={'pcode': str})
same_rows = table[['pcode', 'year', 'month'] + features].merge(test[['pcode', 'year', 'month'] + features], on=['pcode', 'year', 'month'], suffixes=('', '_test'))
print(len(same_rows), 'rows found in both')
differences = {f: (same_rows[f] - same_rows[f + '_test']).abs().max() for f in features}
print('largest difference in any feature:', round(max(differences.values()), 6))

2820 rows found in both
largest difference in any feature: 4e-06


#### Observation

The 25 features are in the same order as the model, and there are no missing values. The same 2,820 rows are in `test.csv`, and the feature values agree to within 0.000004. Even so, the risk is not exactly the same as the test-set risk of `XGBoost.ipynb`: the largest difference is 0.019, 22 rows differ by more than 0.001, and 2 of the 2,820 rows have a different alert level (1,171 alerts here against 1,172 on `test.csv`, and 805 Warnings against 806). The csv file of the full table keeps about 9 digits, and some splits of the trees sit exactly at a training value, so a number that is a little different can fall on the other side. Before `month_sin` and `month_cos` were calculated again (Section 4), the largest difference was 0.25 and 74 rows had a different level, so that step was needed. The 2 rows that remain are a known, small limit.

## 6. Predicting the Risk of Next Month

For each row (a county and a month) the model gives the probability that the county floods in the **next** month. This is called the risk. The alert level comes from the two thresholds, and `flood_recorded` is what really happened in the next month (`flood_next_month`).

In [13]:
#flood probability (risk) of the next month for every row
risk = model.predict_proba(X)[:, 1]

In [14]:
#the app table: keys, the month being predicted, risk, alert level and what was recorded
app = table[['pcode', 'county', 'year', 'month']].copy()
app['forecast_year'] = app['year']
app['forecast_month'] = app['month'] + 1
december = app['month'] == 12
app.loc[december, 'forecast_month'] = 1
app.loc[december, 'forecast_year'] = app['year'] + 1
app['risk'] = risk.round(4)
app['level'] = 'No alert'
app.loc[risk >= watch_threshold, 'level'] = 'Watch'
app.loc[risk >= warning_threshold, 'level'] = 'Warning'
app['flood_recorded'] = table['flood_next_month'].astype(int)
app['records_complete'] = ~((app['year'] == 2024) & (app['month'] == 12))
app = app[['pcode', 'county', 'year', 'month', 'forecast_year', 'forecast_month', 'risk', 'level', 'flood_recorded', 'records_complete']]
app.head()

,pcode,county,year,month,forecast_year,forecast_month,risk,level,flood_recorded,records_complete
0,KE001,Mombasa,2020,1,2020,2,0.2547,No alert,0,True
1,KE002,Kwale,2020,1,2020,2,0.2547,No alert,0,True
2,KE003,Kilifi,2020,1,2020,2,0.3355,No alert,0,True
3,KE004,Tana River,2020,1,2020,2,0.2749,No alert,0,True
4,KE005,Lamu,2020,1,2020,2,0.1794,No alert,0,True


In [15]:
#the predictions are compared with the test-set predictions of the model notebooks (test.csv features, same model)
test_risk = pd.Series(model.predict_proba(test[features])[:, 1], index=pd.MultiIndex.from_frame(test[['pcode', 'year', 'month']]))
app_risk = pd.Series(risk, index=pd.MultiIndex.from_frame(app[['pcode', 'year', 'month']]))
test_risk = test_risk.reindex(app_risk.index)
difference = (app_risk - test_risk).abs()
print('largest difference in risk:', round(difference.max(), 4))
print('rows with a difference above 0.001:', (difference > 0.001).sum())
different_level = ((app_risk >= watch_threshold) != (test_risk >= watch_threshold)) | ((app_risk >= warning_threshold) != (test_risk >= warning_threshold))
print('rows with a different alert level:', different_level.sum(), 'of', len(app_risk))
print('alerts here (watch or warning):', (risk >= watch_threshold).sum(), ' warning only:', (risk >= warning_threshold).sum())
print('alerts on test.csv (watch or warning):', (test_risk >= watch_threshold).sum(), ' warning only:', (test_risk >= warning_threshold).sum())

largest difference in risk: 0.0194
rows with a difference above 0.001: 22
rows with a different alert level: 2 of 2820
alerts here (watch or warning): 1171  warning only: 805
alerts on test.csv (watch or warning): 1172  warning only: 806


In [16]:
#the first and last forecast months, and the december 2024 rows
print(app[['year', 'month', 'forecast_year', 'forecast_month']].drop_duplicates().iloc[[0, 1, -2, -1]])
print(app[app['records_complete'] == False].shape[0], 'rows with records_complete = False')

      year  month  forecast_year  forecast_month
0     2020      1           2020               2
47    2020      2           2020               3
2726  2024     11           2024              12
2773  2024     12           2025               1
47 rows with records_complete = False


### 6.1 Predictions With the Live Inputs

The predictions above use the real `flood_this_month` of the data. A **live** forecast does not have it (no source publishes flood records in real time), and it has no live data for three other inputs either. The live forecast (`src/update_forecast.py`) therefore uses stand-ins:

- `flood_this_month` = 0,
- `soil_moisture_l1` = the training mean of the county and calendar month,
- `runoff_rank` and `discharge_rank` = 0.5.

To show what the live forecast **would have said** in 2020 to 2024, the same model is run again on the same rows with the same stand-ins. This gives two new columns, `risk_live` and `level_live`. The app page "Would it have warned us?" uses them, so that the page is as honest as the live forecast. The first set of columns (`risk`, `level`) stays as it was, with the real `flood_this_month`.

In [17]:
#the same rows with the inputs that the live forecast has: no flood records, soil moisture of the training mean, the two ranks at 0.5
soil_mean = train.groupby(['pcode', 'month'])['soil_moisture_l1'].mean().rename('soil_mean').reset_index()
X_live = X.copy()
X_live['flood_this_month'] = 0
X_live['soil_moisture_l1'] = table[['pcode', 'month']].merge(soil_mean, on=['pcode', 'month'], how='left')['soil_mean'].to_numpy()
X_live['runoff_rank'] = 0.5
X_live['discharge_rank'] = 0.5
print('missing values in the live inputs:', X_live.isna().sum().sum())

missing values in the live inputs: 0


In [18]:
#risk and alert level with the live inputs (same model, same two thresholds)
risk_live = model.predict_proba(X_live)[:, 1]
app['risk_live'] = risk_live.round(4)
app['level_live'] = 'No alert'
app.loc[risk_live >= watch_threshold, 'level_live'] = 'Watch'
app.loc[risk_live >= warning_threshold, 'level_live'] = 'Warning'
app[['risk', 'risk_live']].describe().round(3)

,risk,risk_live
count,2820.000,2820.000
mean,0.338,0.324
std,0.212,0.183
min,0.040,0.040
25%,0.170,0.175
50%,0.287,0.286
75%,0.460,0.453
max,0.954,0.778


In [19]:
#recall of the live inputs on the 2,820 rows (the floods recorded in the forecast month that had a watch or warning level), and the alerts
floods = app['flood_recorded'] == 1
for column in ['level', 'level_live']:
    watch_or_warning = app[column] != 'No alert'
    print(column, '| floods', floods.sum(), '| watch or warning', (floods & watch_or_warning).sum(), 'recall', round((floods & watch_or_warning).sum() / floods.sum(), 3),
          '| warning', (floods & (app[column] == 'Warning')).sum(), 'recall', round((floods & (app[column] == 'Warning')).sum() / floods.sum(), 3),
          '| alerts', watch_or_warning.sum(), 'warnings', (app[column] == 'Warning').sum())

level | floods 144 | watch or warning 141 recall 0.979 | warning 130 recall 0.903 | alerts 1171 warnings 805
level_live | floods 144 | watch or warning 127 recall 0.882 | warning 112 recall 0.778 | alerts 1154 warnings 775


#### Observation

With the real `flood_this_month` (`level`) the model finds 141 of the 144 floods (recall 0.979) with a Watch or a Warning, and 130 of them (0.903) with a Warning, as in Section 6.2. With the inputs of the live forecast (`level_live`) it finds 127 of the 144 (0.882) with a Watch or a Warning, and 112 (0.778) with a Warning. So not knowing the flood records costs 14 floods at the Watch level and 18 at the Warning level, and the number of alerts changes only a little (1,154 Watch or Warning alerts against 1,171, and 775 Warnings against 805). These recalls are the same as the ones of the comparison in `XGBoost_no_flood_input.ipynb` (model 3), so the two notebooks agree. The risk with the live inputs is never above 0.78, while the risk with the real input goes up to 0.95.

In [20]:
#saving the predictions for the app
os.makedirs('../app/data', exist_ok=True)
app.to_csv('../app/data/app_predictions.csv', index=False)
print(app.shape)
print(round(os.path.getsize('../app/data/app_predictions.csv') / 1024), 'KB')

(2820, 12)
185 KB


### 6.2 Alerts per Year

The table counts the Watch and Warning alerts by the year of the data (each year has 12 months of 47 counties = 564 rows; "Watch" here means above the watch line and below the warning line). It also shows the floods that were recorded in the forecast month.

In [21]:
#number of rows at each alert level in each year, and floods recorded in the forecast month
alerts = app.groupby(['year', 'level']).size().unstack(fill_value=0)
alerts = alerts[['Warning', 'Watch', 'No alert']]
alerts['floods recorded'] = app.groupby('year')['flood_recorded'].sum()
alerts

level,Warning,Watch,No alert,floods recorded
year,,,,
2020,174,97,293,39
2021,142,104,318,30
2022,28,66,470,3
2023,275,63,226,31
2024,186,36,342,41


In [22]:
#share of the flood months that were on warning, on watch or without an alert
app[app['flood_recorded'] == 1]['level'].value_counts()

level
Warning     130
Watch        11
No alert      3
Name: count, dtype: int64

#### Observation

In the 2,820 rows there are 1,171 alerts: 805 Warnings and 366 Watches. The Warnings per year are 174 (2020), 142 (2021), 28 (2022), 275 (2023) and 186 (2024); the Watches are 97, 104, 66, 63 and 36. 2022 has far fewer alerts than the other years, and it also has only 3 recorded floods against 30 to 41 in the other years. Of the 144 floods recorded in the forecast month, 130 were on Warning, 11 on Watch and 3 had no alert, so 141 of 144 were found (the same recall of 0.979 as the Watch threshold on the test set). The alerts are far more numerous than the floods, so most of them are false alarms, which is the price of missing fewer floods.

## 7. The County Shapes

The map needs the 47 county shapes. The full shapes file of the data pipeline (in the `El Nino Predictor` folder) has more detail than a dashboard needs, so it is simplified with `geopandas`: the boundary lines are redrawn with fewer points, so the file is smaller and the map loads faster. Only `pcode` and `county` are kept. The original file is only read, never changed.

In [23]:
#reading the county shapes of the data pipeline folder (read only)
shapes = gpd.read_file('../../El Nino Predictor/data/reference/kenya_counties_full.geojson')
original_size = os.path.getsize('../../El Nino Predictor/data/reference/kenya_counties_full.geojson')
print(shapes.shape, shapes.crs)
print(round(original_size / 1024), 'KB')

(47, 6) EPSG:4326
2243 KB


In [24]:
#keeping only the pcode and county columns
shapes = shapes[['pcode', 'county', 'geometry']]

In [25]:
#simplifying the shapes with a larger tolerance (in degrees) until the file is under 1 MB
os.makedirs('../app/data', exist_ok=True)
for tolerance in [0.005, 0.01, 0.02, 0.03]:
    simple = shapes.copy()
    simple['geometry'] = shapes['geometry'].simplify(tolerance)
    simple.to_file('../app/data/kenya_counties_simple.geojson', driver='GeoJSON', COORDINATE_PRECISION=4)
    size = os.path.getsize('../app/data/kenya_counties_simple.geojson')
    print('tolerance', tolerance, 'size', round(size / 1024), 'KB')
    if size < 1024 * 1024:
        break

tolerance 0.005 size 115 KB


### 7.1 Turning the Rings Clockwise

The map of the app is drawn by Plotly, which needs the outer ring of every county to run **clockwise**. `geopandas` writes it counter-clockwise, and with that order Plotly fills the whole map frame instead of only the county. So every ring is turned clockwise and the file is saved again.

In [26]:
#turning the outer ring of every county clockwise and saving the file again
def clockwise(geom):
    if geom.geom_type == 'MultiPolygon':
        return MultiPolygon([orient(part, sign=-1.0) for part in geom.geoms])
    return orient(geom, sign=-1.0)
simple['geometry'] = simple['geometry'].apply(clockwise)
simple.to_file('../app/data/kenya_counties_simple.geojson', driver='GeoJSON', COORDINATE_PRECISION=4)
print(round(os.path.getsize('../app/data/kenya_counties_simple.geojson') / 1024), 'KB')

115 KB


### 7.2 Checking the Simplified Shapes

In [27]:
#reading the saved file again and checking it
saved = gpd.read_file('../app/data/kenya_counties_simple.geojson')
print(saved.shape, list(saved.columns))
print('empty shapes:', saved.geometry.is_empty.sum(), ' invalid shapes:', (~saved.geometry.is_valid).sum())
print('pcodes in the shapes but not in the predictions:', set(saved['pcode']) - set(app['pcode']))
print('pcodes in the predictions but not in the shapes:', set(app['pcode']) - set(saved['pcode']))
names = saved.merge(app[['pcode', 'county']].drop_duplicates(), on='pcode', suffixes=('_shape', '_app'))
print('county names that differ:', (names['county_shape'] != names['county_app']).sum())
first = saved.geometry.iloc[0]
first = first.geoms[0] if first.geom_type == 'MultiPolygon' else first
print('outer ring runs clockwise:', not first.exterior.is_ccw)
print('file size:', round(os.path.getsize('../app/data/kenya_counties_simple.geojson') / 1024), 'KB')

(47, 3) ['pcode', 'county', 'geometry']
empty shapes: 0  invalid shapes: 0
pcodes in the shapes but not in the predictions: set()
pcodes in the predictions but not in the shapes: set()
county names that differ: 0
outer ring runs clockwise: True
file size: 115 KB


In [28]:
#number of points before and after the simplification
def count_points(geometry_series):
    return sum(len(g.exterior.coords) + sum(len(i.coords) for i in g.interiors)
               for geom in geometry_series for g in (geom.geoms if geom.geom_type == 'MultiPolygon' else [geom]))
print('points before', count_points(shapes.geometry))
print('points after', count_points(saved.geometry))

points before 107059
points after 5192


#### Observation

The original shapes file is 2,243 KB with 107,059 points. The first tolerance tried (0.005 degrees, about 500 m) already gives a file of 115 KB with 5,192 points, so the loop stopped there. Plotly needs the outer rings to run clockwise, so they are turned in Section 7.1 (the saved file is checked again in Section 7.2). The file has all 47 counties with `pcode` and `county`, no empty or invalid shapes, the same `pcode` values and the same county names as the predictions. A 500 m simplification is not visible on a map of the whole country.

## 8. Summary of the App Files

- **Model:** the final XGBoost model, loaded from `models/xgboost.joblib`, with the Watch and Warning thresholds from `models/xgboost_settings.json`.
- **Rows:** January 2020 to December 2024 of `model_table_full.csv` (47 counties x 60 months). 2025 is left out because its flood records are almost empty and the model uses `flood_this_month` as a feature.
- **Fix:** `county_flood_rate` is replaced with the training-only rate of each county from `train.csv`, the same values the model was trained with. `month_sin` and `month_cos` are calculated again, because the csv file rounds the September value of `month_cos` and a split of the trees sits exactly there.
- **Features:** exactly the 25 features of the model, in order, with no missing values.
- **Output 1:** `app/data/app_predictions.csv` with the risk, level, forecast month, flood recorded and `records_complete`, and `risk_live` and `level_live` (the same model with the inputs of the live forecast, Section 6.1).
- **Output 2:** `app/data/kenya_counties_simple.geojson`, the 47 counties with `pcode` and `county`, under 1 MB.
- **Private only:** both files are made from the EM-DAT-derived table. `app_predictions.csv` has the flood records of the forecast month, so it may only be committed to a private repository (see `.gitignore` and the README).

The app (`app/app.py`) only reads these two files.